# 04 — Évaluation et comparaison des modèles

**Projet** : FakeNews Analyzer — DevComplex  
**Objectif** : Comparer tous les modèles entraînés, générer le rapport final.

**Exécuter après** : `01_baseline_tfidf.ipynb` et/ou `02_nlp_classical.ipynb`

---

In [ ]:
# ============================================================
# Fichier  : 04_evaluation.ipynb
# Rôle     : Évaluation comparative de tous les modèles
# Version  : V1 (local)
# Projet   : FakeNews Analyzer — DevComplex
# Auteur   : DevComplex
# ============================================================

import sys, os, json
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

MODELS_DIR = '../04_models'
REPORTS_DIR = '../09_data/reports'
os.makedirs(REPORTS_DIR, exist_ok=True)

print('Imports OK')

## Section 1 — Chargement des métadonnées

In [ ]:
metadata_path = os.path.join(MODELS_DIR, 'metadata.json')

if not os.path.exists(metadata_path):
    print('⚠ Aucun metadata.json trouvé. Exécuter les notebooks de modélisation.')
else:
    with open(metadata_path) as f:
        metadata = json.load(f)
    
    df_meta = pd.DataFrame(metadata).T.reset_index().rename(columns={'index': 'model'})
    df_meta = df_meta.sort_values('f1_score', ascending=False)
    
    print('=== MODÈLES DISPONIBLES ===')
    print(df_meta[['model', 'accuracy', 'f1_score', 'auc_roc', 'trained_at']].to_string(index=False))

## Section 2 — Visualisation comparative

In [ ]:
if 'df_meta' in dir():
    metrics = ['accuracy', 'f1_score', 'auc_roc']
    df_plot = df_meta.copy()
    for col in metrics:
        df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    colors = ['#00e5ff', '#00ccaa', '#ff3333', '#ffaa00', '#aa00ff']
    
    for ax, metric in zip(axes, metrics):
        vals = df_plot[metric].fillna(0)
        bars = ax.bar(df_plot['model'], vals, color=colors[:len(df_plot)])
        ax.set_title(metric.replace('_', ' ').title(), fontsize=12)
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='x', rotation=30)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'\n✓ Graphique sauvegardé : {REPORTS_DIR}/model_comparison.png')

## Section 3 — Rapport final

In [ ]:
if 'df_meta' in dir():
    best = df_meta.iloc[0]
    print(f'=== MEILLEUR MODÈLE ===')
    print(f'  Modèle    : {best["model"]}')
    print(f'  Accuracy  : {best["accuracy"]}')
    print(f'  F1-score  : {best["f1_score"]}')
    print(f'  AUC-ROC   : {best["auc_roc"]}')
    print(f'  Entraîné  : {best.get("trained_at", "N/A")}')
    
    report_path = os.path.join(REPORTS_DIR, 'final_report.json')
    with open(report_path, 'w') as f:
        json.dump({'best_model': best.to_dict(), 'all_models': metadata}, f, indent=2)
    print(f'\n✓ Rapport sauvegardé : {report_path}')